In [1]:
from pathlib import Path
from glob import glob
import re
import os

import pandas as pd
import numpy as np

In [2]:
from bikipy.border.parallelogram.classes import ParallelogramBorder
from bikipy.border.triangular import TriangularBorder
from bikipy.behaviour.y_maze.experiment import YMaze
from bikipy.readers import DeepLabCutReader

In [3]:
WORKING_DIR = Path("C:/Users/Can/Projects/Neuroscience/bikipy/examples/data/")
DATA_DIR = Path("C:/Users/Can/Projects/Neuroscience/Imen/data/y_maze/master's")
EXP_ID_FINDER = re.compile("\d+")

BORDER_IMG_PATH = WORKING_DIR / "images" / "y_maze" / "master's.png"
assert BORDER_IMG_PATH.exists(), f"The image file doesn't exist in {BORDER_IMG_PATH}"
border_img_path_str = str(BORDER_IMG_PATH)


In [4]:
borders = (
    ParallelogramBorder(
        base=[[281.46403344, 180.58152957], [257.53029444, 222.75049829]],
        apex=[[145.83951243, 103.08180329], [121.90577343, 140.69196457]],
        guiding_image=border_img_path_str,
        label="A",
    ),
    ParallelogramBorder(
        base=[[280.32433158, 181.72123143], [306.53747429, 223.89020015]],
        apex=[[420.50766001, 100.80239957], [444.44139901, 144.11107015]],
        guiding_image=border_img_path_str,
        label="B",
    ),
    ParallelogramBorder(
        base=[[256.39059258, 222.75049829], [306.53747429, 223.89020015]],
        apex=[[255.25089072, 384.58816201], [305.39777244, 384.58816201]],
        guiding_image=border_img_path_str,
        label="C",
    ),
)
center = TriangularBorder(
    base_a=(257.1623376623377, 223.39610389610385),
    base_b=(280.538961038961, 183.13636363636357),
    apex=(302.6168831168832, 223.39610389610385),
    label="X",
)

In [5]:
CM_PER_PIXEL = np.linalg.norm(center.base_a - center.base_b) / 5

In [6]:
data_dict = {}
for subdir in os.listdir(str(DATA_DIR)):
    data_dict[subdir] = {}
    for file_path in glob(os.path.join(str(DATA_DIR / subdir), "**.h5")):
        exp_id = EXP_ID_FINDER.findall(Path(file_path).stem)[0]
        print(exp_id)
        dlc_data = DeepLabCutReader.from_hdf(
            file_path, (640, 480), midpoint_groups=(("left_ear", "right_ear"),)
        )
        data_dict[subdir][exp_id] = YMaze(
            dlc_data["mid-left_ear-right_ear"],
            borders,
            center,
            14,
            CM_PER_PIXEL,
            exp_id
        )

10
11
12
13
14
15
16
17
18
19
1
20
21
22
23
24
25
26
27
28
29
2
30
31
32
33
34
35
36
37
38
39
3
40
41
42
43
44
45
46
47
48
49
4
50
51
52
53
54
55
56
57
58
59
5
60
61
62
63
64
65
66
67
68
69
6
70
71
72
73
74
75
76
77
78
79
7
80
81
82
83
84
85
86
87
88
89
8
90
91
92
93
9
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
94
95
96
97
98
99


C:\Users\Can\Projects\Neuroscience\bikipy\bikipy\border\base.py:120: UserWarning: Border C has coordinate overlap with other borders
  warn(f"Border {border.label} has coordinate overlap with other borders")


In [7]:
before = YMaze.export_to_dataframe(data_dict["0_before"].values())
after = YMaze.export_to_dataframe(data_dict["1_after"].values())

In [8]:
with pd.ExcelWriter("C:/Users/Can/Projects/Neuroscience/bikipy/examples/data/master's.xlsx") as writer:
    before.to_excel(writer, sheet_name="Before")
    after.to_excel(writer, sheet_name="After")
